# EDA DD

Tujuan:
1. Baca satu file backtest MT5.
2. Agregasi per minggu kerja (Senin-Jumat), ambil DD maksimum tiap minggu.
3. Analisis threshold modal dari DD maksimum mingguan.

In [ ]:
from pathlib import Path
import csv
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = Path(r"C:\Users\user\Downloads\EA MT5\BackTest2020\300.csv")
THRESHOLD = 20000  # threshold = modal

print("Input file:", INPUT_FILE)
print("Threshold modal:", THRESHOLD)


In [ ]:
INPUT_HEADER_TOKENS = {"<DATE>", "DATE", "<BALANCE>", "BALANCE", "<EQUITY>", "EQUITY"}

def is_header_row(row):
    joined_upper = " ".join(str(cell).strip().upper() for cell in row)
    return any(token in joined_upper for token in INPUT_HEADER_TOKENS)

def detect_delimiter(sample: str) -> str:
    try:
        return csv.Sniffer().sniff(sample, delimiters="\t,;").delimiter
    except csv.Error:
        return "\t"

def to_float(value: str):
    value = str(value).strip().replace(" ", "")
    if not value:
        return None
    try:
        return float(value)
    except ValueError:
        try:
            return float(value.replace(",", "."))
        except ValueError:
            return None

def parse_row(row):
    if not row:
        return None
    cleaned = [str(c).strip() for c in row if c is not None]
    if not cleaned or is_header_row(cleaned):
        return None

    if len(cleaned) >= 5:
        date_str, time_str = cleaned[0], cleaned[1]
        balance_str, equity_str = cleaned[2], cleaned[3]
    elif len(cleaned) >= 4:
        parts = cleaned[0].split()
        if len(parts) < 2:
            return None
        date_str, time_str = parts[0], parts[1]
        balance_str, equity_str = cleaned[1], cleaned[2]
    else:
        return None

    balance = to_float(balance_str)
    equity = to_float(equity_str)
    if balance is None or equity is None:
        return None

    return date_str, time_str, balance, equity

def read_rows(path: Path):
    raw = path.read_bytes()
    if raw.startswith(b"\xff\xfe") or raw.startswith(b"\xfe\xff"):
        text = raw.decode("utf-16", errors="replace")
    else:
        text = raw.decode("utf-8-sig", errors="replace")

    delimiter = detect_delimiter(text[:4096])
    reader = csv.reader(text.splitlines(), delimiter=delimiter)
    for row in reader:
        parsed = parse_row(row)
        if parsed is not None:
            yield parsed

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {INPUT_FILE}")

rows = list(read_rows(INPUT_FILE))
df = pd.DataFrame(rows, columns=["date", "time", "balance", "equity"])
df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], errors="coerce")
df["dd"] = df["balance"] - df["equity"]
df = df[df["dd"].notna() & df["datetime"].notna()].reset_index(drop=True)

# Senin=0 ... Minggu=6, ambil hari kerja saja
df = df[df["datetime"].dt.dayofweek <= 4].copy()

# Key minggu dimulai dari Senin
df["week_start"] = (df["datetime"] - pd.to_timedelta(df["datetime"].dt.dayofweek, unit="D")).dt.normalize()

# Dalam 1 minggu (Senin-Jumat), ambil DD maksimum
weekly = (
    df.groupby("week_start", as_index=False)["dd"]
      .max()
      .rename(columns={"dd": "weekly_max_dd"})
)

analysis_df = weekly.copy()

print(f"Rows valid (harian/intraday): {len(df):,}")
print(f"Jumlah minggu kerja         : {len(analysis_df):,}")
print(f"Max weekly DD               : {analysis_df['weekly_max_dd'].max():,.2f}")
analysis_df.head()

In [ ]:
breach = analysis_df["weekly_max_dd"] > THRESHOLD
breach_df = analysis_df[breach].copy()

print(f"Threshold modal : {THRESHOLD:,.2f}")
print(f"Breach count    : {breach.sum():,}")
print(f"Breach %        : {breach.mean() * 100:.2f}%")

if len(breach_df) > 0:
    print(f"Avg excess DD   : {(breach_df['dd'] - THRESHOLD).mean():,.2f}")
    print(f"Worst excess DD : {(breach_df['dd'] - THRESHOLD).max():,.2f}")
else:
    print("Tidak ada DD yang melewati threshold.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(analysis_df["weekly_max_dd"].dropna(), bins=30, alpha=0.4, label="Weekly Max DD (All)")
tail = analysis_df[analysis_df["weekly_max_dd"] >= THRESHOLD]["weekly_max_dd"].dropna()
if len(tail) > 0:
    tail_counts, _, _ = plt.hist(tail, bins=40, alpha=0.85, label=f"DD >= {THRESHOLD}")
    y_max = max(1.0, tail_counts.max() * 1.15)
    plt.ylim(0, y_max)
plt.xlim(THRESHOLD, 100000)
plt.axvline(THRESHOLD, color="red", linestyle="--", linewidth=1.5, label=f"Threshold={THRESHOLD}")
plt.title("Histogram Weekly Max DD (Senin-Jumat)")
plt.xlabel("Weekly Max DD")
plt.ylabel("Jumlah Minggu")
plt.legend()
plt.grid(alpha=0.25)
plt.show()